# Доходности и статистики

Часовые свечи BTC/ETH/SOL/XRP/DOGE за 3 года из чистого слоя. Вопросы:

1. Насколько разная волатильность у монет и как она меняется во времени?
2. Насколько глубоки просадки и сколько длится восстановление?
3. Насколько хвосты толще нормального распределения и что это значит для VaR?
4. Насколько монеты коррелируют (есть ли диверсификация)?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as st

from ohlcv_lab.stats import (
    available_symbols, drawdown, load_closes, log_returns, rolling_vol, summary,
)

CLEAN = "../data/clean"   # путь относительно папки notebooks/
INTERVAL = "60"

# Порядок и цвета фиксированы: цвет следует за монетой, а не за её местом в рейтинге.
COLORS = {"BTCUSDT": "#2a78d6", "ETHUSDT": "#eb6834", "SOLUSDT": "#1baf7a",
          "XRPUSDT": "#eda100", "DOGEUSDT": "#e87ba4"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.figsize": (11, 4.5), "figure.dpi": 110, "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "lines.linewidth": 1.6, "legend.frameon": False,
})

symbols = [s for s in COLORS if s in available_symbols(INTERVAL, CLEAN)]
closes = load_closes(symbols, INTERVAL, CLEAN)
returns = closes.apply(log_returns)
closes.shape

## Сводная таблица

`var_1%` — исторический квантиль: в 1% часов потеря была больше. `var_1%_normal` — то же
по нормальной модели с теми же средним и σ. `tail_share` — доля часов с движением > 4σ;
у нормального распределения это 0.006%.

In [ ]:
table = summary(closes, INTERVAL)
cols = ["ann_vol", "sharpe", "skew", "excess_kurtosis", "var_1%", "var_1%_normal",
        "tail_share", "max_drawdown", "peak", "trough", "recovery"]
table[cols].style.format({
    "ann_vol": "{:.0%}", "sharpe": "{:.2f}", "skew": "{:.2f}", "excess_kurtosis": "{:.1f}",
    "var_1%": "{:.2%}", "var_1%_normal": "{:.2%}", "tail_share": "{:.2%}", "max_drawdown": "{:.0%}",
})

## 1. Скользящая волатильность (30 дней, в годовых)

Волатильность не постоянна: она кластеризуется. Спокойные месяцы сменяются бурными, и у всех
монет одновременно — это и есть общий рыночный фактор.

In [ ]:
vol = pd.DataFrame({s: rolling_vol(returns[s], INTERVAL, "30D") for s in symbols})

fig, ax = plt.subplots()
for s in symbols:
    ax.plot(vol.index, vol[s], color=COLORS[s], label=s.removesuffix("USDT"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title("Скользящая годовая волатильность, окно 30 дней", loc="left")
ax.legend(ncol=5, loc="upper left")
plt.show()

## 2. Просадки

Просадка — насколько цена ниже предыдущего максимума. Ось общая для всех панелей,
чтобы глубину можно было сравнивать на глаз.

In [ ]:
fig, axes = plt.subplots(len(symbols), 1, figsize=(11, 8), sharex=True, sharey=True)
for ax, s in zip(axes, symbols):
    dd = drawdown(closes[s])
    ax.fill_between(dd.index, dd, 0, color=COLORS[s], alpha=0.35, linewidth=0)
    ax.plot(dd.index, dd, color=COLORS[s], linewidth=1)
    worst = dd.idxmin()
    ax.annotate(f"{dd.min():.0%}", (worst, dd.min()), xytext=(0, -11), textcoords="offset points",
                ha="center", fontsize=8, color="#52514e")
    ax.set_ylim(-1, 0.05)
    ax.set_ylabel(s.removesuffix("USDT"), rotation=0, ha="right", va="center")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
axes[0].set_title("Просадка от исторического максимума", loc="left")
plt.show()

## 3. Толстые хвосты

Гистограмма стандартизированных часовых доходностей BTC против нормального распределения.
Ось Y логарифмическая: в обычном масштабе хвостов не видно вообще.

In [ ]:
s = "BTCUSDT"
r = returns[s].dropna()
z = (r - r.mean()) / r.std()

fig, ax = plt.subplots()
bins = np.linspace(-12, 12, 121)
ax.hist(z, bins=bins, density=True, color=COLORS[s], alpha=0.55, label="BTC, факт")
x = np.linspace(-12, 12, 500)
ax.plot(x, st.norm.pdf(x), color=INK, linewidth=1.2, linestyle="--", label="нормальное")
ax.set_yscale("log")
ax.set_ylim(1e-6, 1)
ax.set_xlabel("доходность за час, в σ")
ax.set_title("BTC: часовые доходности против нормального распределения (лог. шкала)", loc="left")
ax.legend()
plt.show()

beyond = (z.abs() > 4).sum()
print(f"движений > 4σ: {beyond} из {len(z)}; нормальная модель ждёт {len(z) * 2 * st.norm.sf(4):.1f}")

## 4. Корреляции

Корреляции часовых логарифмических доходностей. Высокие (0.6–0.8) значения означают, что
портфель из этих пяти монет диверсифицирован слабо: в обвал падает всё вместе.

In [ ]:
corr = returns.corr()
labels = [s.removesuffix("USDT") for s in corr.columns]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr, cmap="Blues", vmin=0.5, vmax=1)  # шкала от 0.5: иначе различий не видно
ax.set_xticks(range(len(labels)), labels)
ax.set_yticks(range(len(labels)), labels)
ax.grid(False)
for i in range(len(labels)):
    for j in range(len(labels)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if v > 0.8 else INK)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Корреляция часовых доходностей", loc="left")
plt.show()

## Выводы

- Годовая волатильность от ~47% (BTC) до ~92% (DOGE); у всех монет она меняется во времени и
  растёт одновременно.
- Эксцесс 9–28: движения > 4σ случаются примерно в 100+ раз чаще, чем предсказывает нормальная модель.
  Нормальный VaR 1% занижает фактический на 20–30% — для риска используем исторический.
- Максимальные просадки от −54% (BTC) до −86% (DOGE).
- Корреляции 0.63–0.82: диверсификация внутри крипты слабая.
- Средняя доходность за час примерно в 100 раз меньше σ: Sharpe на 3 годах — грубая оценка,
  сильно зависящая от выбранного периода.